In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parents[1]))

import joblib
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from sklearn.metrics import (
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

from src.training.train import load_and_prepare_data
from src.training.train_mlp import split_train_val_test
from src.utils.config import (
    MLP_VAL_SIZE,
    OPERATIONAL_THRESHOLD,
    RANDOM_STATE,
    TEST_SIZE,
    get_models_dir,
)

# Configuração visual
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print(f'Threshold operacional: {OPERATIONAL_THRESHOLD}')
print(f'Random state: {RANDOM_STATE}')

Threshold operacional: 0.11
Random state: 42


In [2]:
# carrega dataset
X, y = load_and_prepare_data()

# split 70/15/15
X_train, X_val, X_test, y_train, y_val, y_test = split_train_val_test(
    X, y,
    test_size=TEST_SIZE * 0.75,
    val_size=MLP_VAL_SIZE,
    random_state=RANDOM_STATE,
)

print(f'Tamanho do test set: {len(X_test)} clientes')
print(f'Churn rate no test set: {y_test.mean():.2%}')

2026-04-26 17:29:12,108 - INFO - Carregando dataset de C:\Users\gfrei\Documents\1. FIAP\3. FASE1\9. TECH_CHALLENGE\tech-challenge-fase01\data\raw\WA_Fn-UseC_-Telco-Customer-Churn.csv
2026-04-26 17:29:12,190 - INFO - Dados carregados com sucesso: 7043 linhas, 21 colunas
2026-04-26 17:29:12,191 - INFO - Limpando dados (sem cálculo de estatísticas globais)...
2026-04-26 17:29:12,209 - INFO - Coluna 'TotalCharges' convertida para numérico; 11 valores coagidos para NaN.
2026-04-26 17:29:12,224 - INFO - Coluna 'TotalCharges': 11 valores nulos (0.16%) -- serão tratados no pipeline sklearn.
2026-04-26 17:29:12,287 - INFO - Nenhum registro duplicado encontrado.
2026-04-26 17:29:12,315 - INFO - Limpeza concluída: 7043 linhas, 21 colunas, 11 valores nulos restantes (tratados no pipeline)
2026-04-26 17:29:12,326 - INFO - Aplicando feature engineering...
2026-04-26 17:29:12,329 - INFO - Colunas 'Partner' e 'Dependents' encontradas. Criando 'FamilyStatus'.
2026-04-26 17:29:12,461 - INFO - Variável '

Tamanho do test set: 1057 clientes
Churn rate no test set: 26.49%


In [5]:
# configura MLflow para carregar modelo do Registry
mlflow.set_tracking_uri('sqlite:///../../mlflow.db')

# carrega preprocessor
preprocessor = joblib.load(get_models_dir() / 'mlp_preprocessor.pkl')

# aplica preprocessor no test set
X_test_prep = preprocessor.transform(X_test)
print(f'Shape após preprocessing: {X_test_prep.shape}')

# carrega MLP do MLflow Registry
mlp_model = mlflow.pytorch.load_model(
    'models:/mlp.pt/Production',
    map_location='cpu'
)
mlp_model.eval()

print('Modelo MLP carregado do MLflow Registry')

Shape após preprocessing: (1057, 60)


MlflowException: Registered Model with name=mlp.pt not found